# The Parcel-Picking Robot: Image Segmentation with U-Nets

## The situation

You run operations at a parcel sorting hub. A robotics vendor wants to install a robot arm on one of your conveyor belts, picking cardboard boxes off the belt so your staff don't have to.

A camera looks down at the belt. For every image, the robot has to know **exactly which pixels belong to a box**, so it knows where to place its suction gripper. Round items (rolls of tape, tins, small drums) must be left alone: the gripper can't hold them.

The vendor's demo looked flawless. Before you sign a contract for every belt in the building, you want answers to four questions:

- Does the model work on **our** belt, or only in the vendor's lab?
- If it falls short, is the fix a **bigger model** or **better training photos**?
- How do we even measure whether the robot "sees" a box correctly?
- Is it ready for our **second hub**, where the cameras are older?

You'll build the model yourself and find out.

> 🧩 **Key concept:** This task is called **image segmentation**: labelling every single pixel of an image. The robot doesn't just need to know *that* there's a box; it needs the box's exact outline.

# U-Net Exercise: Segmenting Rectangles from Circles

In this exercise, you will build and compare **U-Nets for image segmentation**. The coding exercise is to be completed within 60 minutes. The lines of code which are to be developed are clearly marked with 🎯 TODO

In the images, the **rectangles** are the boxes the robot should pick and the **circles** are the round items it must leave alone. Here's the plan:

In [1]:
#@title 🗺️ Roadmap: where this notebook is going (double-click to view the code) { display-mode: "form" }
import uuid
from IPython.display import HTML, display

def show_roadmap():
    uid = "rm" + uuid.uuid4().hex[:8]
    steps = [
        ("🧪", "The vendor's demo", "Build a small U-Net and train it on 300 clean lab photos.", "🎯 TODO 1"),
        ("📏", "Measure it", "Score the predictions with IoU and Dice.", "🎯 TODO 2"),
        ("🏭", "Pilot day", "Test the same model on realistic images from your belt.", None),
        ("🧠", "The vendor's offer", "Build a bigger U-Net and train it on the same lab photos.", "🎯 TODO 3"),
        ("📦", "Train on your belt", "Train both U-Nets on 300 realistic images.", None),
        ("🌙", "The challenge set", "Your second hub: what do you expect will happen?", "⭐ Bonus"),
    ]
    cards = ""
    for i, (icon, title, text, todo) in enumerate(steps):
        badge = f'<div class="todo">{todo}</div>' if todo else ""
        cards += f"""
        <div class="card" style="animation-delay:{0.18 * i:.2f}s">
          <div class="bar"><div class="fill" style="animation-delay:{0.18 * i + 0.3:.2f}s"></div></div>
          <div class="head"><span class="num">{i + 1}</span><span class="icon">{icon}</span></div>
          <div class="title">{title}</div>
          <div class="text">{text}</div>
          {badge}
        </div>"""

    display(HTML(f"""
    <style>
      #{uid} {{ font-family: system-ui, "Segoe UI", Roboto, sans-serif; color: #1f2333;
               border-radius: 18px; border: 1px solid #ecebff; padding: 18px 20px 20px;
               background: linear-gradient(135deg, #f6f8ff, #fbf5ff); }}
      #{uid} .label {{ font-weight: 700; font-size: 16px; margin-bottom: 12px; color: #4b3f8f; }}
      #{uid} .row {{ display: flex; flex-wrap: wrap; gap: 12px; }}
      #{uid} .card {{ box-sizing: border-box; flex: 1 1 140px; background: #fff; border: 1px solid #e6e8ee; border-radius: 14px;
                     padding: 12px 14px 14px; opacity: 0; animation: {uid}In 0.5s ease forwards; }}
      #{uid} .bar {{ height: 4px; border-radius: 4px; background: #eef0f6; overflow: hidden; margin-bottom: 10px; }}
      #{uid} .fill {{ height: 100%; width: 0; background: linear-gradient(90deg, #667eea, #764ba2);
                     animation: {uid}Fill 0.6s ease forwards; }}
      #{uid} .head {{ display: flex; align-items: center; justify-content: space-between; }}
      #{uid} .num {{ width: 28px; height: 28px; border-radius: 50%; display: inline-flex; align-items: center;
                    justify-content: center; color: #fff; font-weight: 700; font-size: 14px;
                    background: linear-gradient(135deg, #667eea, #764ba2); }}
      #{uid} .icon {{ font-size: 24px; }}
      #{uid} .title {{ font-weight: 700; font-size: 15px; margin: 8px 0 4px; }}
      #{uid} .text {{ font-size: 13.5px; line-height: 1.4; color: #3d4256; }}
      #{uid} .todo {{ display: inline-block; margin-top: 8px; font-size: 12px; font-weight: 600; color: #5a3d8f;
                     background: #f1ecff; border-radius: 999px; padding: 2px 9px; }}
      @keyframes {uid}In {{ from {{ opacity: 0; transform: translateY(8px); }} to {{ opacity: 1; transform: none; }} }}
      @keyframes {uid}Fill {{ to {{ width: 100%; }} }}
      @media (prefers-reduced-motion: reduce) {{
        #{uid} .card {{ animation: none; opacity: 1; }}
        #{uid} .fill {{ animation: none; width: 100%; }}
      }}
    </style>
    <div id="{uid}">
      <div class="label">🗺️ Roadmap: from the vendor's demo to your second hub</div>
      <div class="row">{cards}</div>
    </div>
    """))

show_roadmap()

## ⚡ Before you start: switch on the GPU

In Colab, go to **Runtime → Change runtime type** and choose a **GPU** (for example T4). The notebook trains four models, and a GPU makes that much faster. The imports cell below prints which one you're using.

## Setup for local users (VS Code, Jupyter)
Google Colab already has all required libraries installed. If you run this notebook locally, the cell below installs them into your current environment. On Colab it does nothing.

In [ ]:
#@title Setup: install libraries (double-click to view the code) { display-mode: "form" }
# Installs the required libraries only when NOT running on Google Colab
if 'google.colab' not in str(get_ipython()):
    %pip install numpy matplotlib torch scikit-image

## Imports

In [ ]:
#@title Setup: imports, GPU check and seeding (double-click to view the code) { display-mode: "form" }
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
!wget -q -O shapes_data.py https://raw.githubusercontent.com/eth-bmai-hs26/w3-cx-public/main/unet/shapes_data.py
from shapes_data import print_training_data_set, ShapesDataset, IMAGE_SIZE

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", "GPU" if device == "cuda" else "CPU (switch on the GPU for faster training)")

def set_seed(seed):
    # Fix all random numbers, so a cell gives the same result every time it runs
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## The lab photos

These are the vendor's **lab photos**: one box (the bright square) and one round item (the grey circle) on a clean, empty belt. Under each photo is the answer we want the model to produce: white where there's a box, black everywhere else.

We make 300 photos to train on and 100 new ones to test on. The model never sees the test photos while it learns.

In [ ]:
#@title Make the lab photos (double-click to view the code) { display-mode: "form" }
set_seed(1)
clean_train = ShapesDataset(300)
set_seed(2)
clean_test = ShapesDataset(100)

print_training_data_set(clean_train)

## U-Net architecture

A **U-Net** is a neural network for **image segmentation**: it takes an image and predicts which pixels belong to which class.

### The general structure

- **Channels**: are like layers showing different features. Grayscale images have 1 channel; color images have 3. The network creates more channels to detect edges, textures, and shapes.
- **Encoder (compress)**: shrinks the image step by step, halving its size each time. Like zooming out: the network learns *what* is in the image, but loses the fine detail of *exactly where* the edges are.
- **Bottleneck**: the most compressed representation, at the bottom of the U. It knows "there's a box roughly here, and that's a tin", but only on a tiny, coarse version of the image.
- **Decoder (decompress)**: enlarges the image step by step, back to full size.
- **Output**: a score for every pixel, saying how sure the network is that the pixel belongs to a box.

### Skip connections: what makes it a U-Net

Encoder, bottleneck and decoder on their own make an **autoencoder** (you met them in the autoencoder exercise). Its decoder has to rebuild the full-size image from the coarse bottleneck alone, so the outlines come out blurry.

A U-Net adds **skip connections**: bridges across the U. At each level on the way back up, the decoder receives the sharp, full-detail version the encoder saved on the way down, and combines the two:

- from the deep path: **what** it is (a box, not a tin);
- from the skip connection: **exactly where** its edges are.

> 🧩 **Key concept:** Skip connections let a U-Net *understand* the image at a coarse level and still *draw* its answer pixel by pixel. For the robot: the deep path recognises the box, and the skip connections tell the gripper exactly where its edges are.

In the code, a skip connection is the `torch.cat([...])` line in the decoder. It sticks the encoder's saved channels onto the decoder's, which is why a decoder block takes **twice as many channels** in as it gives out (for example 16 + 16 = 32 → 16).

**🎯 TODO 1:** complete the small U-Net below by filling in the lines marked `# 🎯 TODO`.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder path
        # 🎯 TODO: Complete the code below
        self.enc1 = DoubleConv(None, None)  # 🎯 TODO: first encoder block
        self.enc2 = DoubleConv(None, None)  # 🎯 TODO: second encoder block

        self.pool = nn.MaxPool2d(None)  # 🎯 TODO: downsampling layer
        # Bottleneck
        self.bottleneck = DoubleConv(16, 32)  # Bottom of the U, feature expansion: 16 → 32 channels

        # Decoder path
        self.up2 = nn.ConvTranspose2d(32, 16, 2, 2)     # Upsample from 32 → 16 channels, doubles spatial dimensions
        self.dec2 = DoubleConv(32, 16)                  # After concatenation with skip connection: 16+16=32 → 16

        self.up1 = nn.ConvTranspose2d(16, 8, 2, 2)      # Upsample from 16 → 8 channels
        self.dec1 = DoubleConv(16, 8)                   # After concatenation with skip connection: 8+8=16 → 8

        # Final output layer
        self.out = nn.Conv2d(8, 1, 1)   # 1x1 convolution to reduce to desired output channels (e.g., segmentation mask)


    def forward(self, x):
        # Encoder
        # 🎯 TODO: Apply first double conv
        e1 = None  # 🎯 TODO
        # 🎯 TODO: Pool e1 to reduce size, then apply second double conv
        e2 = None  # 🎯 TODO

        # Bottleneck
        b = self.bottleneck(self.pool(e2))   # Pool again, then double conv at the bottom of the U

        # Decoder
        d2 = self.up2(b)                   # Upsample bottleneck features
        d2 = torch.cat([d2, e2], dim=1)    # Concatenate skip connection from encoder (same spatial size)
        d2 = self.dec2(d2)                 # Process concatenated features with double conv

        d1 = self.up1(d2)                   # Upsample
        d1 = torch.cat([d1, e1], dim=1)     # Concatenate skip connection from first encoder
        d1 = self.dec1(d1)                  # Process with double conv

        # Output
        return self.out(d1)                 # Final 1x1 convolution to produce output

# U-Net Training Explained

**Inputs:**  
- The network receives an **image**.  

**Ground truth mask:**  
- For training, each image has a **mask** showing the correct labeling of pixels (e.g., 1 for rectangle, 0 for background).  

**Prediction:**  
- The U-Net outputs a **score for every pixel**: a map the same size as the input, saying how sure it is that each pixel belongs to a box. Pixels it is more than 50% sure about form the **predicted mask**.

**Loss calculation:**  
- Compare the model's **scores** with the ground truth mask using a **loss function** (in this scenario we use Binary Cross Entropy loss).  
- The loss measures how close the prediction is to the truth.  

**Backpropagation:**  
- The network updates its weights to **minimize the loss**, improving mask predictions over time.  


The function below trains a model and draws its training and test loss. You don't need to change it, so its code is folded away: **double-click the cell** to remind yourself how training works, step by step.

Every model in this notebook is trained with exactly the same settings (40 epochs), so the comparisons are fair.

In [ ]:
#@title ⚙️ The training function: train_model() (double-click to remind yourself how training works) { display-mode: "form" }
criterion = nn.BCEWithLogitsLoss()

def train_model(model_class, train_dataset, test_dataset, n_rounds=40, seed=0):
    # Train a fresh model of the given kind on train_dataset, then plot its loss on both datasets
    try:
        set_seed(seed)
        model = model_class().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,
                                  generator=torch.Generator().manual_seed(seed))
        test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

        train_losses = []
        test_losses = []

        for epoch in range(n_rounds):
            # ---- Training phase ----
            train_epoch_loss = 0
            model.train()

            for x, y in train_loader:
                x, y = x.to(device), y.to(device)

                optimizer.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                optimizer.step()

                train_epoch_loss += loss.item()

            train_losses.append(train_epoch_loss / len(train_loader))

            # ---- Testing phase ----
            test_epoch_loss = 0
            model.eval()

            with torch.no_grad():
                for x, y in test_loader:
                    x, y = x.to(device), y.to(device)
                    test_epoch_loss += criterion(model(x), y).item()

            test_losses.append(test_epoch_loss / len(test_loader))

            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1:02d} | Train loss: {train_losses[-1]:.4f} | Test loss: {test_losses[-1]:.4f}")
    except Exception as e:
        raise NotImplementedError("Something is missing above. Complete all 🎯 TODO sections before training.") from e

    plt.plot(train_losses, marker="o", label="Train Loss")
    plt.plot(test_losses, marker="o", label="Test Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.yscale("log")
    plt.title("Training and test loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    return model

## Train the small U-Net on the lab photos

In [ ]:
small_lab_model = train_model(SmallUNet, clean_train, clean_test)

## Evaluating the Segmentation

The `segmentation_metrics()` function measures how closely the U-Net's predicted mask matches the ground-truth rectangle mask. It calculates two important segmentation metrics: **Intersection over Union (IoU)** and **Dice score**. Both measure the overlap between the predicted and correct regions. IoU is calculated as the intersection of the predicted and true regions divided by their union, while Dice gives twice the intersection divided by the total size of the two regions. Both metrics range from 0 to 1, where **1 means a perfect segmentation and 0 means no overlap**. Therefore, higher IoU and Dice scores indicate better segmentation.

**First, a yes/no answer per pixel.** IoU and Dice count box pixels, so every pixel first needs a clear answer: box or not. We cut the model's scores at 50%: every pixel the model is more than 50% sure about counts as box (the `threshold=0.5` in `segmentation_metrics()`). Every white area in the prediction pictures is made with the same cut.

**IoU in words:**

$$\text{IoU} = \frac{\text{box pixels the model got right}}{\text{all pixels that are box in the prediction, the truth, or both}}$$

**Dice in words:**

$$\text{Dice} = \frac{2 \times \text{box pixels the model got right}}{\text{box pixels in the prediction} + \text{box pixels in the truth}}$$

<details>
<summary><b>Show a worked example</b></summary>

A box covers 1,000 pixels. The model marks 1,000 pixels as "box": 800 of them really are the box, and 200 are on a tin next to it. So it gets 800 pixels right, misses 200 pixels of the box, and wrongly marks 200 pixels of the tin.

- IoU = 800 ÷ (800 right + 200 missed + 200 wrong) = 800 ÷ 1,200 ≈ **0.67**
- Dice = 2 × 800 ÷ (1,000 + 1,000) = 1,600 ÷ 2,000 = **0.80**

Same prediction, two scores: Dice is always a bit more generous than IoU, but the two rise and fall together.

</details>

The `evaluate_model()` function applies these measurements to the entire dataset. It runs the U-Net in evaluation mode, calculates the BCE loss, IoU, and Dice score for each batch, and then returns their average values. This allows us to evaluate the model using both its **loss** and its **segmentation quality**.

**🎯 TODO 2:** complete the three lines in `evaluate_model()` marked `# 🎯 TODO`.


In [ ]:

def segmentation_metrics(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    intersection = (preds * targets).sum(dim=(1, 2, 3))
    pred_area = preds.sum(dim=(1, 2, 3))
    target_area = targets.sum(dim=(1, 2, 3))

    union = pred_area + target_area - intersection

    iou = (intersection + 1e-8) / (union + 1e-8)
    dice = (2 * intersection + 1e-8) / (pred_area + target_area + 1e-8)

    return iou.mean().item(), dice.mean().item()


def evaluate_model(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_iou = 0.0
    total_dice = 0.0
    n_batches = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            # 🎯 TODO: get the model logits
            logits = None  # 🎯 TODO
            # 🎯 TODO: calculate the BCE loss
            loss = criterion(None, None)  # 🎯 TODO
            # 🎯 TODO: complete the following line
            iou, dice = None, None  # 🎯 TODO

            total_loss += loss.item()
            total_iou += iou
            total_dice += dice
            n_batches += 1

    return (
        total_loss / n_batches,
        total_iou / n_batches,
        total_dice / n_batches,
    )


In [ ]:
#@title Helpers: print scores and show predictions (double-click to view the code) { display-mode: "form" }
# Helpers to print the scores and show predictions (no need to change)

def report(model, **test_sets):
    # Print loss, IoU and Dice of a model on each named test set
    for name, dataset in test_sets.items():
        loader = DataLoader(dataset, batch_size=16, shuffle=False)
        try:
            loss, iou, dice = evaluate_model(model, loader, criterion)
        except Exception as e:
            raise NotImplementedError("Something is missing in evaluate_model(). Complete 🎯 TODO 2 before scoring the models.") from e
        print(f"{name:>9s} images | Loss: {loss:.4f} | IoU: {iou:.3f} | Dice: {dice:.3f}")


def clean_axis(ax, label=None):
    # Hide the frame of one picture; if a label is given, write it on the left of the row
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    if label:
        ax.set_ylabel(label.replace(", ", "\n"), rotation=0, ha="right", va="center", fontsize=10, labelpad=8)


def show_predictions(models, dataset, n=4):
    # Show n images: the input, then one row per model, then the correct answer
    rows = [("Input", "input")] + list(models.items()) + [("Ground truth", "truth")]
    fig, axes = plt.subplots(len(rows), n, figsize=(1.8 * n + 1.6, 1.75 * len(rows)))

    for model in models.values():
        model.eval()

    with torch.no_grad():
        for i in range(n):
            img, mask = dataset[i]
            x = img.unsqueeze(0).to(device)

            for r, (label, what) in enumerate(rows):
                if what == "input":
                    picture = img[0]
                elif what == "truth":
                    picture = mask[0]
                else:
                    picture = torch.sigmoid(what(x))[0, 0].cpu() > 0.5

                axes[r, i].imshow(picture, cmap="gray", vmin=0, vmax=1)
                clean_axis(axes[r, i], label if i == 0 else None)

    plt.tight_layout()
    plt.show()

## How well does it do on new lab photos?

In [ ]:
#@title 📊 Scores and predictions: small U-Net on new lab photos (double-click to view the code) { display-mode: "form" }
report(small_lab_model, lab=clean_test)
show_predictions({"Small U-Net, lab photos": small_lab_model}, clean_test)

**Reading the result:** This is the demo you were shown. Trained on lab photos, the small U-Net finds the box and ignores the round item on new lab photos, with an IoU close to 1. Impressive, but keep in mind where those photos came from.

# Pilot Day: Noisy and Ambiguous Images

Time for the pilot. We point the camera at a real belt in your hub, and the footage looks nothing like the lab:

- several parcels and round items on the belt at once, sometimes touching;
- a box resting against a tin, so their outlines overlap;
- lighting that changes along the belt, so some boxes look dark brown, almost as dim as the tins;
- a camera that adds grain to every image.

The robot's job hasn't changed: outline the boxes, ignore everything else. In the images, the goal is still the same: **segment the rectangles**.

> **Does a U-Net trained on lab photos still work when the input changes?**

## Creating a harder dataset

For the advanced dataset, the image is generated with several distractors and target objects.

The mask contains **only the rectangles**. Circles are distractors, even when they overlap with a rectangle.

We make 300 belt images to train on, as many as the lab photos, and 100 to test on. Only the kind of image changes, not the amount.

You do not need to touch the code below. Just run! It creates the same number of images as before, but much more difficult ones.

In [ ]:
#@title Belt image generator (double-click to view the code) { display-mode: "form" }
# Advanced image generator
from skimage.draw import rectangle, disk

def generate_advanced_sample(
    image_size=IMAGE_SIZE,
    min_circles=2,
    max_circles=5,
    min_rectangles=1,
    max_rectangles=3,
    noise_std_range=(0.05, 0.15),
):
    image = np.zeros((image_size, image_size), dtype=np.float32)
    mask = np.zeros((image_size, image_size), dtype=np.float32)

    # Add multiple circles (distractors)
    n_circles = np.random.randint(min_circles, max_circles + 1)

    for _ in range(n_circles):
        r = np.random.randint(8, 21)
        cy, cx = np.random.randint(r, image_size - r, size=2)

        rr, cc = disk((cy, cx), r)

        # Circle brightness overlaps with some rectangle brightnesses,
        # making the segmentation task less trivial.
        brightness = np.random.uniform(0.20, 0.70)
        image[rr, cc] = np.maximum(image[rr, cc], brightness)

    # Add multiple rectangles (targets)
    n_rectangles = np.random.randint(min_rectangles, max_rectangles + 1)

    for _ in range(n_rectangles):
        h = np.random.randint(15, 36)
        w = np.random.randint(15, 36)

        ry = np.random.randint(0, image_size - h)
        rx = np.random.randint(0, image_size - w)

        rr, cc = rectangle(start=(ry, rx), extent=(h, w))

        # Rectangle brightness varies from image to image.
        brightness = np.random.uniform(0.40, 1.00)
        image[rr, cc] = np.maximum(image[rr, cc], brightness)

        # The rectangle is always part of the target mask.
        mask[rr, cc] = 1.0

    # Add random Gaussian noise to the entire image.
    noise_std = np.random.uniform(*noise_std_range)
    noise = np.random.normal(0, noise_std, image.shape).astype(np.float32)
    image = np.clip(image + noise, 0.0, 1.0)

    return image, mask


In [ ]:
#@title Belt image dataset (double-click to view the code) { display-mode: "form" }
# A fixed dataset: samples are generated once so evaluation is stable.
from torch.utils.data import Dataset
class AdvancedShapesDataset(Dataset):
    def __init__(self, n_samples, **generator_kwargs):
        self.images = []
        self.masks = []

        for _ in range(n_samples):
            img, mask = generate_advanced_sample(**generator_kwargs)
            self.images.append(img)
            self.masks.append(mask)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.images[idx], dtype=torch.float32).unsqueeze(0),
            torch.tensor(self.masks[idx], dtype=torch.float32).unsqueeze(0)
        )


In [ ]:
set_seed(3)
messy_train = AdvancedShapesDataset(300, noise_std_range=(0.05, 0.15))
set_seed(4)
messy_test = AdvancedShapesDataset(100, noise_std_range=(0.05, 0.15))

print("Training examples:", len(messy_train))
print("Test examples:", len(messy_test))

In [ ]:
#@title 📊 Show some belt images (double-click to view the code) { display-mode: "form" }
# Visualise the harder training examples

def show_advanced_samples(dataset, n=6):
    fig, axes = plt.subplots(3, n, figsize=(1.8 * n + 1.6, 5.2))

    for i in range(n):
        img, mask = dataset[i]

        axes[0, i].imshow(img[0], cmap="gray", vmin=0, vmax=1)
        axes[1, i].imshow(mask[0], cmap="gray", vmin=0, vmax=1)
        axes[2, i].imshow(img[0], cmap="gray", vmin=0, vmax=1)
        axes[2, i].imshow(mask[0], cmap="Reds", alpha=0.35)

        for r, label in enumerate(["Input", "Rectangle mask", "Overlay"]):
            clean_axis(axes[r, i], label if i == 0 else None)

    plt.tight_layout()
    plt.show()

show_advanced_samples(messy_train)


## First test: does the lab model work on your belt?

This is pilot day: we take the vendor's model, trained only on lab photos, and run it on footage from your belt **without retraining it**.

This is a direct test of generalisation: the input distribution has changed.

In [ ]:
#@title 📊 Scores and predictions: small U-Net on the belt images (double-click to view the code) { display-mode: "form" }
report(small_lab_model, lab=clean_test, belt=messy_test)
show_predictions({"Small U-Net, lab photos": small_lab_model}, messy_test)

**Reading the result:** On the lab photos the model was close to perfect. On your belt, the IoU and Dice drop sharply and the loss rises.

Look at the predictions. Which boxes does the model find, and which does it miss? In the lab photos every box was bright white and every round item grey, so the model may have learned "the brightest thing is a box" rather than "a square is a box".

**Why does this matter in business?** A box the model misses is a parcel the robot skips, and someone on your team has to sort it by hand. The demo told you nothing about that.

# The Vendor's Offer: A Bigger U-Net

The vendor's answer: "Our larger model will handle it." A larger model needs more computing power on every robot, so it costs more to buy and to run. Before you pay for it, let's test the claim.

**🎯 TODO 3:** make the U-Net deeper by adding a **third encoder level** and the matching decoder level. Fill in every line marked `# 🎯 TODO` in the cell below.

In [ ]:
class LargeUNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder path
        self.enc1 = DoubleConv(1, 8)   # First double conv: input 1 channel → 8 channels
        self.enc2 = DoubleConv(8, 16)  # Second double conv: 8 → 16 channels

        # 🎯 TODO: add a third encoder layer
        self.enc3 = None  # 🎯 TODO: third encoder block

        self.pool = nn.MaxPool2d(2)    # Max pooling with 2x2 window, reduces spatial dimensions by half

        # Bottleneck
        #self.bottleneck = DoubleConv(16, 32)  # Bottom of the U, feature expansion: 32 → 64 channels
        # 🎯 TODO: update the bottleneck to match the output of the third encoder
        self.bottleneck = None  # 🎯 TODO: bottleneck

        # Decoder path
        # 🎯 TODO: add a third decoder layer
        self.up3 = None  # 🎯 TODO: third upsampling layer
        self.dec3 = None  # 🎯 TODO: third decoder block

        self.up2 = nn.ConvTranspose2d(32, 16, 2, 2)  # Upsample from 32 → 16 channels, doubles spatial dimensions
        self.dec2 = DoubleConv(32, 16)              # After concatenation with skip connection: 16+16=32 → 16

        self.up1 = nn.ConvTranspose2d(16, 8, 2, 2)   # Upsample from 16 → 8 channels
        self.dec1 = DoubleConv(16, 8)               # After concatenation with skip connection: 8+8=16 → 8

        # Final output layer
        self.out = nn.Conv2d(8, 1, 1)   # 1x1 convolution to reduce to desired output channels (e.g., segmentation mask)


    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)                # Apply first double conv
        e2 = self.enc2(self.pool(e1))    # Pool e1 to reduce size, then apply second double conv#
        # 🎯 TODO: add a third encoder layer
        e3 = None  # 🎯 TODO
        b = self.bottleneck(self.pool(e3))
        # Bottleneck
        # b = self.bottleneck(self.pool(e2))   # Pool again, then double conv at the bottom of the U

        # Decoder
        # 🎯 TODO: add a third decoder layer
        d3 = None  # 🎯 TODO
        d3 = torch.cat([d3, e3], dim=1)
        d3 = None  # 🎯 TODO

        d2 = self.up2(d3)                   # Upsample bottleneck features
        #d2 = self.up2(b)                   # Upsample bottleneck features
        d2 = torch.cat([d2, e2], dim=1)    # Concatenate skip connection from encoder (same spatial size)
        d2 = self.dec2(d2)                 # Process concatenated features with double conv

        d1 = self.up1(d2)                   # Upsample
        d1 = torch.cat([d1, e1], dim=1)     # Concatenate skip connection from first encoder
        d1 = self.dec1(d1)                  # Process with double conv

        # Output
        return self.out(d1)                 # Final 1x1 convolution to produce output

## Train the bigger U-Net on the same lab photos

Same photos, same training settings: the only difference is the size of the model.

In [ ]:
large_lab_model = train_model(LargeUNet, clean_train, clean_test)

## Does the bigger model pass the pilot?

In [ ]:
#@title 📊 Scores and predictions: both lab-trained U-Nets on the belt images (double-click to view the code) { display-mode: "form" }
report(large_lab_model, lab=clean_test, belt=messy_test)
show_predictions({"Small U-Net, lab photos": small_lab_model,
                  "Large U-Net, lab photos": large_lab_model}, messy_test)

**Reading the result:** On the lab photos, the bigger model is just as close to perfect as the small one: lab photos are too easy to tell the two apart. On your belt, compare its scores with the small model's. The bigger model does somewhat better, but it still misses far too many boxes for the robot to go live, and it sometimes mistakes a bright round item for a box.

**Business takeaway:** The bigger model costs more, and it doesn't solve the problem. Is the problem really the model?

# Train on Your Own Belt

**The fix:** instead of a bigger model, change what the model learns from. We train both U-Nets again, this time on 300 images taken in your hub conditions: the same number of images and the same training settings as before.

In real life, this means paying for photos from your own belt to be collected and labelled: a **data** cost, not a **model** cost.

In [ ]:
small_belt_model = train_model(SmallUNet, messy_train, messy_test)

In [ ]:
large_belt_model = train_model(LargeUNet, messy_train, messy_test)

**Look at the two training curves.** For the small U-Net, the training and test loss stay close together all the way. For the large U-Net, towards the end of training the training loss keeps falling while the test loss levels off. The bigger model is starting to learn its 300 training images a little *too* well, instead of learning what a box looks like in general: this is called **overfitting**. Here it's only mild, but a growing gap between the two curves is the warning sign to look for.

## How do the belt-trained models do?

In [ ]:
#@title 📊 Scores and predictions: both belt-trained U-Nets (double-click to view the code) { display-mode: "form" }
print("Small U-Net, trained on belt images")
report(small_belt_model, lab=clean_test, belt=messy_test)
print()
print("Large U-Net, trained on belt images")
report(large_belt_model, lab=clean_test, belt=messy_test)

show_predictions({"Small U-Net, belt images": small_belt_model,
                  "Large U-Net, belt images": large_belt_model}, messy_test)

**Reading the result:** Compare these scores with the lab-trained models on the belt images. The **small** U-Net trained on belt images beats the **large** U-Net trained on lab photos by a wide margin. And it still does well on the lab photos: learning from harder images cost nothing on the easy ones.

**Business takeaway:** The cheap model with the right data beats the expensive model with the wrong data. A bigger model is worth considering once the data is right, not instead of it. When a vendor proposes an improvement, ask whether it changes the model or the data, and whether the data looks like your operation.

## 🔍 A Look Inside the U-Net: Small and Large

So far we've only looked at what goes in and what comes out. Let's follow one belt image through the U-Nets trained on belt images, level by level.

**Look at both:** use the dropdown in the cell below to switch between the small and the large U-Net. The picture redraws by itself.

Each level has many channels. For each level, we show the one channel that lines up best with the boxes.

In [ ]:
#@title 📊 A look inside the U-Net: pick the small or the large one (double-click to view the code) { display-mode: "form" }
import torch.nn.functional as F
from matplotlib.patches import ConnectionPatch

def look_inside(model, dataset, index=2):
    # Show what one image looks like at each level of the U (works for the small and the large U-Net)
    img, mask = dataset[index]
    model.eval()
    depth = 3 if hasattr(model, "enc3") else 2

    # Record the output of every level while the image passes through the U-Net
    encoders = [f"enc{i}" for i in range(1, depth + 1)]
    decoders = [f"dec{i}" for i in range(1, depth + 1)]
    seen = {}
    hooks = [getattr(model, name).register_forward_hook(
                 lambda module, inputs, output, name=name: seen.__setitem__(name, output[0].cpu()))
             for name in encoders + ["bottleneck"] + decoders]
    with torch.no_grad():
        how_sure = torch.sigmoid(model(img.unsqueeze(0).to(device)))[0, 0].cpu()
    for hook in hooks:
        hook.remove()

    # Each level has many channels; show the one that lines up best with the boxes
    def best_channel(features):
        target = F.interpolate(mask.unsqueeze(0), size=features.shape[-2:], mode="area")[0, 0].flatten()
        target = target - target.mean()
        channels = features.flatten(1)
        channels = channels - channels.mean(1, keepdim=True)
        match = (channels @ target) / (channels.norm(dim=1) * target.norm() + 1e-8)
        best = match.abs().argmax()
        return features[best] * match[best].sign()

    def size(name):
        return f"{seen[name].shape[-1]}×{seen[name].shape[-1]} pixels"

    top = ([("Input", img[0])] +
           [(f"Encoder level {i}\n" + size(f"enc{i}"), best_channel(seen[f"enc{i}"])) for i in range(1, depth + 1)] +
           [("Bottleneck\n" + size("bottleneck"), best_channel(seen["bottleneck"]))])
    bottom = ([("The mask", how_sure > 0.5, "gray"),
               ("Output: how sure\n(dark = no box, bright = box)", how_sure, "magma")] +
              [(f"Decoder level {i}\n" + size(f"dec{i}"), best_channel(seen[f"dec{i}"]), "magma") for i in range(1, depth + 1)])

    # Top row starts one column to the right, so each decoder level sits under its encoder level
    n_cols = depth + 3
    fig, axes = plt.subplots(2, n_cols, figsize=(2.8 * n_cols, 7))
    for c, (title, picture) in enumerate(top):
        axes[0, c + 1].imshow(picture, cmap="gray" if c == 0 else "magma", interpolation="nearest")
        axes[0, c + 1].set_title(title, fontsize=11)
    for c, (title, picture, cmap) in enumerate(bottom):
        vmax = 1 if c < 2 else None
        axes[1, c].imshow(picture, cmap=cmap, vmin=0 if c < 2 else None, vmax=vmax, interpolation="nearest")
        axes[1, c].set_xlabel(title, fontsize=11)
    for ax in axes.flat:
        clean_axis(ax)
    axes[0, 0].axis("off")
    axes[1, n_cols - 1].axis("off")
    fig.text(0.04, 0.75, "The way down:\nshrink", fontsize=12, fontweight="bold", va="center")
    fig.text(0.04, 0.5, "The way up:\nenlarge", fontsize=12, fontweight="bold", va="center")
    plt.tight_layout(h_pad=4)

    def arrow(ax_from, xy_from, ax_to, xy_to, color, label=None):
        fig.add_artist(ConnectionPatch(xy_from, xy_to, "axes fraction", "axes fraction", axesA=ax_from, axesB=ax_to,
                                       arrowstyle="-|>", color=color, lw=3, mutation_scale=22))
        if label:
            x = (ax_from.get_position().x0 + ax_from.get_position().x1) / 2
            y = (ax_from.get_position().y0 + ax_to.get_position().y1) / 2
            fig.text(x + 0.008, y, label, color=color, fontsize=10, fontweight="bold", va="center")

    for c in range(2, depth + 2):    # skip connections: straight down from encoder to decoder level
        arrow(axes[0, c], (0.5, -0.02), axes[1, c], (0.5, 1.02), "#667eea", "skip connection")
    arrow(axes[0, n_cols - 1], (0.5, -0.02), axes[1, n_cols - 2], (1.02, 0.5), "#999999")    # bottleneck back up
    plt.show()

model_to_show = "Small U-Net"  #@param ["Small U-Net", "Large U-Net"] {run: "auto"}
look_inside(small_belt_model if model_to_show == "Small U-Net" else large_belt_model, messy_test)

**Reading the picture (small U-Net):**

- **The way down.** At encoder level 1 the picture is still sharp, and everything bright lights up: boxes and tins alike. Each level is half the size of the one before, and blurrier.
- **The bottleneck.** At 32×32 pixels the picture is coarse and blocky, but now mostly the **boxes** light up: the tins have faded. The U-Net knows *what* is a box, but only roughly *where*.
- **The way up.** The decoder enlarges the picture again, and the skip connections bring back the sharp detail from the way down. By decoder level 1, the box edges are sharp again and the tins are dark.
- **The last step.** The output is a score for every pixel: how sure the U-Net is that the pixel belongs to a box. Only then is it turned into the black-and-white mask. Where the colour is in between, the model is unsure, usually along edges and where shapes overlap.

This is the skip-connection idea from the start of the notebook, now visible: the bottleneck decides *what* is a box, and the skip connections supply *where* its edges are.

**Now switch to the large U-Net.** It has one more level on each side, so its bottleneck is even coarser: only 16×16 pixels. But each deeper level sees a larger area of the image at once, and by encoder level 3 it has already separated the boxes from the tins. On the way up, its decoder level 1 shows only the boxes, and its final outline is a little cleaner than the small U-Net's. That's the small edge in its scores, made visible.

## Compare the four models

Look at the table below and discuss:

- Which change helped more: a bigger model, or training images that look like the real job?
- What kinds of images are still difficult for the best model?
- Would you improve the architecture, the data, or the loss function next?

And the questions you'd have to answer as the manager:

- What would you ask the vendor for first: more photos, a bigger model, or a different way of reporting results?
- Which mistake is more expensive at your hub: a skipped box or a grabbed tin? Would you move the 50% cut, and in which direction?

In [ ]:
#@title 📊 Summary table: every model on the lab and belt images (double-click to view the code) { display-mode: "form" }
models = {
    "Small U-Net, lab photos": small_lab_model,
    "Large U-Net, lab photos": large_lab_model,
    "Small U-Net, belt images": small_belt_model,
    "Large U-Net, belt images": large_belt_model,
}

def summary_table(test_sets):
    # Print the IoU (and loss) of every model on every test set
    print(f"{'Trained on':25s} | " + " | ".join(f"{name + ' IoU (loss)':>21s}" for name in test_sets))
    for model_name, model in models.items():
        cells = []
        for dataset in test_sets.values():
            loss, iou, dice = evaluate_model(model, DataLoader(dataset, batch_size=16), criterion)
            cells.append(f"{iou:13.3f} ({loss:.3f})")
        print(f"{model_name:25s} | " + " | ".join(cells))

summary_table({"lab": clean_test, "belt": messy_test})

# ⭐ Bonus: The Challenge Set, Your Second Hub

*This part is optional: do it if you have time left.*

Now the rollout question. Your second hub has older cameras and busier belts on the night shift. The models have never seen a single image from there.

The challenge set contains:

- more shapes
- stronger noise
- more overlap

> 💭 **Think first:** before you run the cells below, write down what you expect. Which of the four models will hold up best? Will the belt-trained models drop a little or a lot?

In [ ]:
set_seed(5)
challenge_dataset = AdvancedShapesDataset(
    100,
    min_circles=3,
    max_circles=6,
    min_rectangles=1,
    max_rectangles=4,
    noise_std_range=(0.15, 0.25)
)

In [ ]:
#@title 📊 Predictions of all four models on the challenge set (double-click to view the code) { display-mode: "form" }
show_predictions(models, challenge_dataset)

In [ ]:
#@title 📊 Summary table, now with the challenge set (double-click to view the code) { display-mode: "form" }
summary_table({"lab": clean_test, "belt": messy_test, "challenge": challenge_dataset})

Discuss:

- How did the models do on the challenge set, compared with what you expected?
- Would you approve the rollout to the second hub based on these numbers?

# Final Challenge

You have now seen that making a segmentation problem harder is not only about making the neural network deeper.

The main challenge is **generalisation**.

Once you have completed the main exercise, try changing **one parameter at a time** and observe how it affects the model's performance. (Treat this as a bonus exercise)

Experiment with:

- **Noise level:** Try different `noise_std_range` values.
- **Number of training examples:** Increase or decrease the dataset size.
- **Number of epochs:** Train the model for longer or shorter (the `n_rounds` setting of `train_model`).


### Key takeaway

> A good segmentation model should not only perform well on the images it was trained on. It should continue to produce useful masks when the input changes.

## 💬 Debrief

### What you learned about U-Nets

1. **A U-Net answers pixel by pixel.** The encoder shrinks the image to understand *what* is in it; the decoder enlarges it back to full size, so there is an answer for every pixel.
2. **Skip connections are what make it a U-Net.** The bottleneck knows *what* is a box but only roughly *where*; the skip connections carry the sharp detail across the U, so the outline lands on the right pixels. Without them, it would be an autoencoder with blurry outlines.
3. **The output is a score, not a mask.** The U-Net says how sure it is about every pixel; the black-and-white mask comes from a cut afterwards. Training works on the scores, while IoU and Dice judge the mask.
4. **A deeper U-Net sees more at once.** Each extra level halves the picture again and lets the model take in a larger area, which helped the large U-Net separate boxes from tins earlier. It also costs more, and it was the first to show signs of overfitting.

### What you'd tell the vendor

1. **A lab demo isn't evidence.** A model that's perfect on clean photos can still miss boxes on a real belt. Test on your own footage before you sign.
2. **Better data beat a bigger model.** The big improvement came from training on photos that look like your operation. A bigger model added only a little on top.
3. **Look at the predictions, not just the scores.** The pictures show *which* boxes are missed, something a single average can't tell you.
4. **Every new site is a new test.** Different cameras and lighting mean the model has to be checked again before it goes live there.